# 08 · Video

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/microsoft/SPARROW-Playground/blob/main/notebooks/08_video.ipynb)

Run detection, or detection plus classification, on frames sampled from a video file.
Video support needs OpenCV: `pip install "sparrow-engine[video]"`.

In [ ]:
# Setup: install sparrow-engine if it is missing (for example on Colab).
# For an NVIDIA GPU, install `sparrow-engine-gpu` instead (same Python API).
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("sparrow_engine") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "sparrow-engine[video]>=0.1.31"])

import sparrow_engine as spe

print("sparrow-engine", spe.__version__, "| device:", spe.active_device())

In [ ]:
# Demo data: downloaded once into ./data and checked against a published checksum.
import hashlib
import os
import shutil
import urllib.request
import zipfile
from pathlib import Path

DATA = Path(os.environ.get("SPP_DATA_DIR", "data")).resolve()
ASSETS = {
    # Pytorch-Wildlife demo assets, CC-BY-4.0, https://zenodo.org/records/15376499
    "demo_data_det.zip": (
        "https://zenodo.org/records/15376499/files/demo_data_det.zip?download=1",
        "md5", "373e15c8e7c8894379a82086a9177763",
    ),
    "demo_data_base.zip": (
        "https://zenodo.org/records/15376499/files/demo_data_base.zip?download=1",
        "md5", "f89b9b76c753d90e4296d62d71cf1732",
    ),
    # "Yellow-Throated Vireo Song" by SwampVids, CC-BY-4.0, Wikimedia Commons
    "vireo.wav": (
        "https://upload.wikimedia.org/wikipedia/commons/4/4f/"
        "Vireo_flavifrons_-_Yellow-Throated_Vireo_Song.wav",
        "sha1", "2b472209c1b18bfc0c225975838e3a34c1e9887a",
    ),
}


def fetch(name):
    """Download an asset (once), verify it, and unzip archives into DATA."""
    url, algo, digest = ASSETS[name]
    target = DATA / name
    DATA.mkdir(parents=True, exist_ok=True)
    if not target.exists():
        print(f"downloading {name} ...")
        part = target.with_name(target.name + ".part")
        req = urllib.request.Request(url, headers={"User-Agent": "sparrow-playground"})
        with urllib.request.urlopen(req) as r, open(part, "wb") as f:
            shutil.copyfileobj(r, f)
        part.rename(target)
    h = hashlib.new(algo)
    with open(target, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    if h.hexdigest() != digest:
        target.unlink()
        raise RuntimeError(f"{name}: checksum mismatch, deleted; run the cell again")
    marker = DATA / f".{name}.extracted"
    if name.endswith(".zip") and not marker.exists():
        with zipfile.ZipFile(target) as z:
            z.extractall(DATA)
        marker.touch()
    return target

In [ ]:
fetch("demo_data_base.zip")  # about 95 MB
video = DATA / "demo_data/videos/opossum_example.MP4"

import cv2
cap = cv2.VideoCapture(str(video))
print(f"{cap.get(cv2.CAP_PROP_FRAME_COUNT):.0f} frames at {cap.get(cv2.CAP_PROP_FPS):.0f} fps")
cap.release()

In [ ]:
# Models are installed into SPARROW_ENGINE_MODEL_DIR (default ~/.sparrow-engine/models).
# Already-installed models are skipped. Inference never downloads anything by itself.
for model_id in ["MDV6-yolov10-c", "AI4G-Amazon-V2"]:
    spe.download_model(model_id)

Sample one frame per second (`every_n=30` at 30 fps) and stop after 10 frames.

In [ ]:
frames = spe.detect_video(video, model="MDV6-yolov10-c", every_n=30, max_frames=10,
                          threshold=0.3)
for f in frames:
    best = max((d.confidence for d in f.result.detections), default=0.0)
    print(f"frame {f.index:4d}  t={f.timestamp_s:5.1f}s  detections={len(f.result.detections)}"
          f"  max confidence={best:.2f}")

## Stream frames with classification

`iter_video` yields results as it goes, so long videos do not fill memory.
`keep_frames=True` returns the decoded frame (BGR, as OpenCV reads it) for drawing.

In [ ]:
import matplotlib.pyplot as plt

shown = []
for f in spe.iter_video(video, model="MDV6-yolov10-c", classifier="AI4G-Amazon-V2",
                        every_n=60, max_frames=4, threshold=0.3, keep_frames=True):
    labels = [(d.classification.label, round(d.classification.confidence, 2))
              for d in f.result.detections if d.classification]
    print(f"t={f.timestamp_s:4.1f}s", labels)
    shown.append(spe.draw_detections(f.frame, f.result))

fig, axes = plt.subplots(1, len(shown), figsize=(5 * len(shown), 3))
for ax, img in zip(axes, shown):
    ax.imshow(img[..., ::-1])  # BGR -> RGB
    ax.axis("off")
plt.tight_layout()

## Write an annotated clip

In [ ]:
out = Path("outputs/08_video")
out.mkdir(parents=True, exist_ok=True)
writer = None
for f in spe.iter_video(video, model="MDV6-yolov10-c", every_n=15, max_frames=20,
                        threshold=0.3, keep_frames=True):
    frame = spe.draw_detections(f.frame, f.result)
    if writer is None:
        h, w = frame.shape[:2]
        writer = cv2.VideoWriter(str(out / "annotated.mp4"), cv2.VideoWriter_fourcc(*"mp4v"),
                                 2, (w, h))
    writer.write(frame)
writer.release()
print(out / "annotated.mp4")